# 🎯 SFDAO - Synthetic Finance Data Auditor & Optimizer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/takurot/sfdao/blob/main/notebooks/sfdao_demo.ipynb)
[![PyPI version](https://badge.fury.io/py/sfdao.svg)](https://badge.fury.io/py/sfdao)

このノートブックでは、SFDAOの主要機能をデモンストレーションします。

## 📋 Contents

1. **インストール** - PyPIからのインストール
2. **データ準備** - サンプルデータのダウンロード
3. **基本監査** - `sfdao audit` コマンドの使用
4. **合成データ生成** - 簡易生成機能
5. **Phase 2 ワークフロー** - 生成 → ガードレール → 監査
6. **Python APIの使用** - プログラマティックな利用
7. **レポート出力** - HTML/PDFレポート

---
## 1. 📦 インストール

In [ ]:
# SFDAOのインストール
!pip install -q sfdao

# バージョン確認
!sfdao --version

---
## 2. 📊 データ準備

デモ用のサンプルデータを準備します。ここでは簡易的なクレジットカード取引データを生成します。

In [ ]:
import pandas as pd
import numpy as np

# 乱数シード固定
np.random.seed(42)

# サンプルの「実データ」を生成
n_samples = 1000

real_data = pd.DataFrame({
    'Time': np.sort(np.random.uniform(0, 172800, n_samples)),
    'V1': np.random.normal(0, 1, n_samples),
    'V2': np.random.normal(0, 1, n_samples),
    'V3': np.random.normal(0, 1, n_samples),
    'V4': np.random.normal(0, 1.5, n_samples),
    'V5': np.random.normal(0, 1, n_samples),
    'Amount': np.abs(np.random.lognormal(3, 1.5, n_samples)),
    'Class': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
})

# CSVとして保存
real_data.to_csv('real_data.csv', index=False)
print(f"✅ Real data created: {len(real_data)} rows")
real_data.head()

---
## 3. 🔬 合成データ生成

SFDAOに含まれるベースライン生成器を使って合成データを生成します。

In [ ]:
# 合成データ生成スクリプトの使用
!python -m sfdao.scripts.generate_test_synthetic_data \
    real_data.csv \
    synthetic_data.csv \
    --n-samples 1000 \
    --random-state 42

In [ ]:
# 生成された合成データを確認
synthetic_data = pd.read_csv('synthetic_data.csv')
print(f"✅ Synthetic data: {len(synthetic_data)} rows")
synthetic_data.head()

---
## 4. 📋 基本監査（CLI）

`sfdao audit` コマンドで合成データの品質を評価します。

In [ ]:
# テキストレポート出力
!sfdao audit \
    --real real_data.csv \
    --synthetic synthetic_data.csv \
    --output report.txt

# レポート内容を表示
with open('report.txt', 'r') as f:
    print(f.read())

In [ ]:
# HTMLレポート出力
!sfdao audit \
    --real real_data.csv \
    --synthetic synthetic_data.csv \
    --output report.html

print("✅ HTML report saved to report.html")

In [ ]:
# Colab上でHTMLレポートを表示
from IPython.display import IFrame, display, HTML

with open('report.html', 'r') as f:
    html_content = f.read()

display(HTML(html_content))

---
## 5. 🐍 Python APIの使用

CLIだけでなく、Python APIからも直接SFDAOの機能を利用できます。

In [ ]:
from sfdao.ingestion.loader import load_csv
from sfdao.ingestion.type_detector import TypeDetector

# データの読み込み
real_df = load_csv('real_data.csv')
synthetic_df = load_csv('synthetic_data.csv')

# 自動型検出
detector = TypeDetector()
schema = detector.detect_all(real_df)

print("📊 Detected Column Types:")
for col, col_type in schema.items():
    print(f"  - {col}: {col_type.value}")

In [ ]:
from sfdao.evaluator.statistical import StatisticalEvaluator

# 統計評価の実行
stat_evaluator = StatisticalEvaluator()
stat_result = stat_evaluator.evaluate(real_df, synthetic_df)

print("📈 Statistical Evaluation Results:")
print(f"  Overall Score: {stat_result.overall_score:.3f}")
print(f"  Mean JS Divergence: {stat_result.mean_js_divergence:.4f}")
print(f"  Mean KS Statistic: {stat_result.mean_ks_statistic:.4f}")

In [ ]:
from sfdao.evaluator.privacy import PrivacyEvaluator

# プライバシー評価の実行
privacy_evaluator = PrivacyEvaluator()
privacy_result = privacy_evaluator.evaluate(real_df, synthetic_df)

print("🔒 Privacy Evaluation Results:")
print(f"  Overall Score: {privacy_result.overall_score:.3f}")
print(f"  Re-identification Risk: {privacy_result.reidentification_risk:.4f}")
print(f"  Mean DCR: {privacy_result.mean_dcr:.4f}")
print(f"  Min DCR: {privacy_result.min_dcr:.4f}")

In [ ]:
from sfdao.evaluator.finance_facts import FinancialFactsChecker

# 金融特化評価（Stylized Facts）
finance_checker = FinancialFactsChecker()
finance_result = finance_checker.evaluate(real_df, synthetic_df)

print("💰 Financial Facts Evaluation:")
print(f"  Overall Score: {finance_result.overall_score:.3f}")
print(f"  Fat Tail Preserved: {finance_result.fat_tail_preserved}")
print(f"  Volatility Clustering: {finance_result.volatility_clustering_preserved}")

---
## 6. ⚙️ 評価スコアの統合

複数の評価結果を統合して総合スコアを計算します。

In [ ]:
from sfdao.evaluator.scorer import Scorer

# スコアラーによる統合評価
scorer = Scorer()
final_score = scorer.calculate_final_score(
    statistical_result=stat_result,
    privacy_result=privacy_result,
    finance_result=finance_result
)

print("\n" + "="*50)
print("🎯 FINAL EVALUATION SUMMARY")
print("="*50)
print(f"  Statistical Score:  {stat_result.overall_score:.3f}")
print(f"  Privacy Score:      {privacy_result.overall_score:.3f}")
print(f"  Finance Score:      {finance_result.overall_score:.3f}")
print(f"  ─────────────────────────────")
print(f"  📊 FINAL SCORE:     {final_score:.3f}")
print("="*50)

---
## 7. 🔄 Phase 2: ワークフロー自動化

`sfdao run` コマンドで、設定ファイルに基づいた一括処理ができます。

In [ ]:
# Phase 2用の設定ファイルを作成
config_yaml = """
generator:
  type: baseline
  n_samples: 500
  random_state: 42

guard:
  rules:
    - column: Amount
      type: range
      min: 0
      max: 50000
    - column: Class
      type: enum
      values: [0, 1]

scenario:
  enabled: false

audit:
  statistical: true
  privacy: true
  finance: true

output:
  format: html
"""

with open('config.yaml', 'w') as f:
    f.write(config_yaml)

print("✅ Config file created: config.yaml")

In [ ]:
# Phase 2 ワークフローの実行
!sfdao run \
    --real real_data.csv \
    --config config.yaml \
    --out-dir output_phase2

In [ ]:
# 出力ファイルの確認
import os

output_dir = 'output_phase2'
if os.path.exists(output_dir):
    print(f"📁 Output files in {output_dir}:")
    for f in os.listdir(output_dir):
        print(f"  - {f}")
else:
    print("⚠️ Output directory not found")

---
## 8. 📊 分布の可視化

実データと合成データの分布を比較します。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

columns_to_plot = ['V1', 'V2', 'V3', 'V4', 'Amount', 'Class']

for ax, col in zip(axes.flatten(), columns_to_plot):
    ax.hist(real_df[col], bins=30, alpha=0.6, label='Real', density=True)
    ax.hist(synthetic_df[col], bins=30, alpha=0.6, label='Synthetic', density=True)
    ax.set_title(f'{col} Distribution')
    ax.legend()
    ax.set_xlabel(col)
    ax.set_ylabel('Density')

plt.tight_layout()
plt.suptitle('Real vs Synthetic Data Distribution', y=1.02, fontsize=14)
plt.savefig('distribution_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Distribution comparison saved to distribution_comparison.png")

---
## 9. 🧹 クリーンアップ

In [ ]:
# 生成したファイルのリスト
import os

files = [
    'real_data.csv', 'synthetic_data.csv', 
    'report.txt', 'report.html', 
    'config.yaml', 'distribution_comparison.png'
]

print("📁 Generated files:")
for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✅ {f} ({size:,} bytes)")

# 必要に応じてコメントアウトを解除してクリーンアップ
# for f in files:
#     if os.path.exists(f):
#         os.remove(f)
# print("\n🧹 Files cleaned up")

---
## 📚 Next Steps

- **ドキュメント**: [GitHub Repository](https://github.com/takurot/sfdao)
- **PyPI**: `pip install sfdao`
- **高度な機能**: `pip install sfdao[deep]` でCTGAN対応版をインストール

### 主な機能

| 機能 | 説明 |
|------|------|
| 統計評価 | KS検定、JS Divergenceによる分布比較 |
| プライバシー評価 | 再識別リスク、DCR (Distance to Closest Record) |
| 金融特化評価 | Fat Tail検出、Volatility Clustering検証 |
| 自動型検出 | Numeric, Categorical, Datetime, PIIの自動分類 |
| レポート生成 | HTML/PDF/TXTフォーマット対応 |
| ガードレール | ルールベースの制約チェック |
| シナリオ注入 | scale/shift/clip/outlier等のテスト |

---

**Thank you for using SFDAO!** 🎉